In [1]:
import warnings
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
import scipy.sparse
from preprocessing import preprocess_data
from basic_models import random_forest_classifier
from basic_models import gradient_boosting_classifier
from basic_models import LSTM_preds
from basic_models import logistic_reg_model
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.gaussian_process.kernels import RBF, ExpSineSquared, DotProduct, WhiteKernel
from sklearn.gaussian_process.kernels import Matern

warnings.filterwarnings("ignore")

2025-03-28 17:25:35.759125: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-28 17:25:35.798318: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-03-28 17:25:35.798350: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-03-28 17:25:35.799461: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-28 17:25:35.806213: I tensorflow/core/platform/cpu_feature_guar

In [2]:
data, X_train_processed, X_test_processed, y_train, y_test = preprocess_data("stores_sales_forecasting 2.csv")
data

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,lag_profit_1,rolling_sales_mean_3,rolling_profit_mean_3,cohort,churn,Order Year,Order Month,Ship Year,Ship Month,Shipping Delay
20,79,US-2014-147606,2014-11-26,2014-12-01,Second Class,JE-15745,Joel Eaton,Consumer,United States,Houston,...,1.2130,316.092000,-42.551067,0,0,2014,11,2014,12,5
39,178,US-2015-101511,2015-11-21,2015-11-23,Second Class,JE-15745,Joel Eaton,Consumer,United States,Newark,...,-14.4750,171.047333,-8.199733,12,0,2015,11,2015,11,2
51,235,US-2017-100930,2017-04-07,2017-04-12,Standard Class,CS-12400,Christopher Schild,Home Office,United States,Tampa,...,-248.2458,370.848833,-116.764600,0,1,2017,4,2017,4,5
54,242,CA-2016-157749,2016-06-04,2016-06-09,Second Class,KL-16645,Ken Lonsdale,Consumer,United States,Chicago,...,-4.6752,202.864333,-160.638733,23,1,2016,6,2016,6,5
55,243,CA-2016-157749,2016-06-04,2016-06-09,Second Class,KL-16645,Ken Lonsdale,Consumer,United States,Chicago,...,-120.5130,64.319000,-42.673000,23,1,2016,6,2016,6,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2116,9963,CA-2015-168088,2015-03-19,2015-03-22,First Class,CM-12655,Corinna Mitchell,Home Office,United States,Houston,...,88.7332,316.008533,12.558933,0,0,2015,3,2015,3,3
2117,9965,CA-2016-146374,2016-12-05,2016-12-10,Second Class,HE-14800,Harold Engle,Corporate,United States,Newark,...,22.9885,51.110000,12.871967,0,0,2016,12,2016,12,5
2118,9981,US-2015-151435,2015-09-06,2015-09-09,Second Class,SW-20455,Shaun Weien,Consumer,United States,Lafayette,...,22.5296,106.393333,15.686000,0,1,2015,9,2015,9,3
2119,9990,CA-2014-110422,2014-01-21,2014-01-23,Second Class,TB-21400,Tom Boeckenhauer,Consumer,United States,Miami,...,41.8608,340.516000,86.550800,0,1,2014,1,2014,1,2


### Possible Columns to delete
Row ID – Just an index, no predictive value.

Order ID / Product ID – Unique per transaction, not customer-specific.

Order Date / Ship Date / Ship Year / Ship Month – These might introduce data leakage if they reveal churn status indirectly.

Customer Name – Not useful for prediction.

Country – If the dataset is only from one country, this can be removed.

Postal Code – Too specific; regional data is likely sufficient.

Product Name – Too granular; category/sub-category is better.

In [3]:
# columns_to_drop = [
#     "Row ID", "Order ID", "Order Date", "Ship Date", "Ship Year", "Ship Month",
#     "Customer Name", "Country", "Postal Code", "Product ID", "Product Name"
# ]

# data = data.drop(columns=columns_to_drop)

In [4]:
X_train_dense = X_train_processed.toarray() if scipy.sparse.issparse(X_train_processed) else X_train_processed
X_test_dense = X_test_processed.toarray() if scipy.sparse.issparse(X_test_processed) else X_test_processed

lstm_out = LSTM_preds(X_train_dense, X_test_dense, y_train)

2025-03-28 17:25:43.958393: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9804 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:3e:00.0, compute capability: 7.5


Epoch 1/20


2025-03-28 17:25:47.182931: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8902
2025-03-28 17:25:47.969586: I external/local_xla/xla/service/service.cc:168] XLA service 0x7f4190322c00 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-03-28 17:25:47.969619: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 2080 Ti, Compute Capability 7.5
2025-03-28 17:25:47.974713: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1743182748.156378    8262 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


11/11 [==============================] - 4s 14ms/step - loss: 0.5414 - accuracy: 0.0000e+00
Epoch 2/20
11/11 [==============================] - 0s 14ms/step - loss: 0.0134 - accuracy: 0.0000e+00
Epoch 3/20
11/11 [==============================] - 0s 15ms/step - loss: -0.4718 - accuracy: 0.0000e+00
Epoch 4/20
11/11 [==============================] - 0s 15ms/step - loss: -0.9031 - accuracy: 0.0000e+00
Epoch 5/20
11/11 [==============================] - 0s 15ms/step - loss: -1.2487 - accuracy: 0.0000e+00
Epoch 6/20
11/11 [==============================] - 0s 15ms/step - loss: -1.5159 - accuracy: 0.0000e+00
Epoch 7/20
11/11 [==============================] - 0s 15ms/step - loss: -1.8968 - accuracy: 0.0000e+00
Epoch 8/20
11/11 [==============================] - 0s 15ms/step - loss: -2.1701 - accuracy: 0.0000e+00
Epoch 9/20
11/11 [==============================] - 0s 15ms/step - loss: -2.6582 - accuracy: 0.0000e+00
Epoch 10/20
11/11 [==============================] - 0s 15ms/step - loss: -3.

In [5]:
# Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_dense)
X_test = scaler.transform(X_test_dense)

### Try different kernels
Tried:
-RBF
-Matern

In [10]:
# Define the kernel: linear + nonlinear
kernel = 1.0 * DotProduct() + 1.0 * RBF(length_scale=1.0)

In [11]:
# Train Guassian Classifier
gp_model = GaussianProcessClassifier(kernel= kernel, random_state=42)
gp_model.fit(X_train, y_train)

GaussianProcessClassifier(kernel=1**2 * DotProduct(sigma_0=1) + 1**2 * RBF(length_scale=1),
                          random_state=42)

In [23]:
# Predict churn probabilities
churn_probs = gp_model.predict_proba(X_test)[:, 1]  # Probability of churn

In [25]:
# Print churn scores for first 5 customers
print(churn_probs[:5]) 

[0.5 0.5 0.5 0.5 0.5]


In [6]:
# from sklearn.gaussian_process.kernels import RBF, Matern, RationalQuadratic, DotProduct, ExpSineSquared

# kernels = [
#     1.0 * RBF(length_scale=1.0),  # Standard smooth kernel
#     1.0 * Matern(length_scale=1.0, nu=0.5),  # Less smooth than RBF
#     1.0 * RationalQuadratic(length_scale=1.0, alpha=1.0),  # Handles varying smoothness
#     1.0 * DotProduct() + 1.0 * RBF(length_scale=1.0),  # Linear + Non-linear
#     1.0 * ExpSineSquared(length_scale=1.0, periodicity=3.0)  # Cyclic patterns
# ]

# for kernel in kernels:
#     print(f"Trying kernel: {kernel}")
#     gpc = GaussianProcessClassifier(kernel=kernel)
#     gpc.fit(X_train, y_train)
#     churn_probs = gpc.predict_proba(X_test)[:, 1]
#     print(churn_probs[:5])  # Print first 5 churn scores

Trying kernel: 1**2 * RBF(length_scale=1)
[0.5 0.5 0.5 0.5 0.5]
Trying kernel: 1**2 * Matern(length_scale=1, nu=0.5)
[0.5 0.5 0.5 0.5 0.5]
Trying kernel: 1**2 * RationalQuadratic(alpha=1, length_scale=1)
[0.61294075 0.61294143 0.61294083 0.61294053 0.61294137]
Trying kernel: 1**2 * DotProduct(sigma_0=1) + 1**2 * RBF(length_scale=1)
[0.59407084 0.68742156 0.57589807 0.56758008 0.67957149]
Trying kernel: 1**2 * ExpSineSquared(length_scale=1, periodicity=3)


LinAlgError: 36-th leading minor of the array is not positive definite